In [19]:
!pip install -q pandas streamlit plotly anthropic requests matplotlib fpdf2 transformers sentencepiece

In [20]:
import os, getpass

os.environ["EMAIL_ADDRESS"] = "worofyousef@gmail.com"
os.environ["EMAIL_APP_PASSWORD"] = getpass.getpass("Gmail App Password: ")
os.environ["RECIPIENT_EMAIL"] = "worofyousef@gmail.com"

Gmail App Password: ··········


In [21]:
!mkdir -p agents reports


In [22]:
import pandas as pd

# Load the GTA V sales data from the uploaded CSV file
df_sales = pd.read_csv('/content/gta_v_worldwide_sales_player_analytics_2013_2026.csv')

# Display the first 5 rows to confirm it loaded correctly
display(df_sales.head())

,transaction_id,year,month,quarter,country,iso3_code,region,platform,game_edition,sales_channel,...,gaming_market_size,population_millions,gdp_per_capita_usd,release_phase,weekend_sales_percentage,weekday_sales_percentage,season,special_event,top_game_category,platform_generation
0,GTA5-100039,2013,9,3,Algeria,DZA,Africa,PS3,Premium Edition,Physical,...,4.82,38.28,5621.76,Launch,41.4,58.6,Autumn,Xbox Sale,Action-Adventure,Gen7
1,GTA5-100005,2013,9,3,Argentina,ARG,South America,PS3,Standard Edition,Physical,...,23.48,42.31,14452.74,Launch,40.8,59.2,Autumn,Xbox Sale,Action-Adventure,Gen7
2,GTA5-100093,2013,9,3,Argentina,ARG,South America,Xbox 360,Premium Edition,Physical,...,23.96,42.31,14748.61,Launch,51.6,48.4,Autumn,Xbox Sale,Action-Adventure,Gen7
3,GTA5-100072,2013,9,3,Armenia,ARM,Asia,PS3,Standard Edition,Physical,...,0.14,2.92,3879.16,Launch,42.3,57.7,Autumn,Xbox Sale,Action-Adventure,Gen7
4,GTA5-100034,2013,9,3,Australia,AUS,Oceania,PS3,Legacy Edition,Physical,...,231.44,23.17,69365.83,Launch,43.4,56.6,Autumn,Xbox Sale,Action-Adventure,Gen7


In [23]:
%%writefile config.py
import os


EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS", "worofyousef@gmail.com")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD", "")
RECIPIENT_EMAIL = os.getenv("RECIPIENT_EMAIL", "worofyousef@gmail.com")
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

DEFAULT_CSV_PATH = "/content/gta_v_worldwide_sales_player_analytics_2013_2026.csv"
DASHBOARD_STATE_PATH = "dashboard_state.json"
REPORTS_DIR = "reports"

Overwriting config.py


In [24]:
%%writefile agents/__init__.py

Overwriting agents/__init__.py


In [25]:
%%writefile agents/retrieval_agent.py
import pandas as pd
import requests


class RetrievalAgent:
    KEYWORD_MAP = {
        "country": ["country", "region", "market"],
        "platform": ["platform", "console"],
        "sales": ["sales", "revenue", "units_sold", "copies_sold"],
        "players": ["players", "active_players", "player_count"],
        "date": ["date", "year", "release_date"],
    }

    def _detect_columns(self, df: pd.DataFrame) -> dict:
        cols_lower = {c.lower(): c for c in df.columns}
        detected = {}
        for field, keywords in self.KEYWORD_MAP.items():
            for kw in keywords:
                match = next((orig for low, orig in cols_lower.items() if kw in low), None)
                if match:
                    detected[field] = match
                    break
        return detected

    def load_sales_data(self, csv_path: str) -> dict:
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        return {"df": df, "columns": self._detect_columns(df)}

    def fetch_exchange_rate(self, base="USD", target="EGP"):
        try:
            resp = requests.get(f"https://api.exchangerate-api.com/v4/latest/{base}", timeout=8)
            resp.raise_for_status()
            return resp.json()["rates"].get(target)
        except Exception:
            return None

    def retrieve_all(self, csv_path: str) -> dict:
        sales = self.load_sales_data(csv_path)
        fx_rate = self.fetch_exchange_rate()
        return {"sales_df": sales["df"], "columns": sales["columns"], "usd_to_egp": fx_rate}

Overwriting agents/retrieval_agent.py


In [26]:
%%writefile agents/analysis_agent.py
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Changed imports

import config


class AnalysisAgent:
    def __init__(self):
        # Initialize Hugging Face model and tokenizer for sequence-to-sequence generation
        self.model_name = "google/flan-t5-large"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)

    def compute_stats(self, df: pd.DataFrame, columns: dict) -> dict:
        stats = {"rows": len(df)}
        sales_col = columns.get("sales")
        country_col = columns.get("country")
        platform_col = columns.get("platform")
        players_col = columns.get("players")

        if sales_col:
            stats["total_sales"] = float(df[sales_col].sum())
            stats["avg_sales"] = float(df[sales_col].mean())
        if country_col and sales_col:
            stats["top_countries"] = (
                df.groupby(country_col)[sales_col].sum().sort_values(ascending=False).head(5).to_dict()
            )
        if platform_col and sales_col:
            stats["top_platforms"] = (
                df.groupby(platform_col)[sales_col].sum().sort_values(ascending=False).head(5).to_dict()
            )
        if players_col:
            stats["total_players"] = float(df[players_col].sum())
            stats["avg_players"] = float(df[players_col].mean())
        return stats

    def generate_insights(self, stats: dict, fx_rate) -> str:
        prompt = f"""Summarize the following game sales and player statistics into a concise executive summary (150-200 words). The summary should cover overall performance, highlight a standout country/platform, make a player-engagement observation, and provide one actionable recommendation.

Stats (JSON): {json.dumps(stats, default=str)}
Current USD to EGP exchange rate: {fx_rate}

Where a headline money figure appears, give it in both USD and EGP.
"""

        # Encode the prompt
        input_ids = self.tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).input_ids

        # Generate text using the model
        # Parameters like max_new_tokens can be adjusted based on desired output length
        outputs = self.model.generate(input_ids, max_new_tokens=250, do_sample=True, temperature=0.7)

        # Decode the generated text
        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return generated_text

Overwriting agents/analysis_agent.py


In [27]:
%%writefile agents/action_agent.py
import json
import os
import datetime
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.application import MIMEApplication

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from fpdf import FPDF

import config


class ActionAgent:
    def __init__(self):
        os.makedirs(config.REPORTS_DIR, exist_ok=True)

    def _make_chart(self, stats: dict, chart_path: str):
        top = stats.get("top_countries") or stats.get("top_platforms")
        if not top:
            return None
        fig, ax = plt.subplots(figsize=(6, 3.5))
        ax.bar(list(top.keys()), list(top.values()), color="#3E7CB1")
        ax.set_title("Top performers by sales")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        fig.savefig(chart_path)
        plt.close(fig)
        return chart_path

    def generate_report(self, stats: dict, insights: str) -> str:
        chart_path = os.path.join(config.REPORTS_DIR, "chart.png")
        self._make_chart(stats, chart_path)

        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 16)
        pdf.cell(0, 10, "GTA V Sales & Player Analytics Report", ln=True)
        pdf.set_font("Helvetica", size=11)
        pdf.ln(4)
        pdf.multi_cell(0, 6, insights)
        pdf.ln(4)
        if os.path.exists(chart_path):
            pdf.image(chart_path, w=170)
        pdf.ln(4)
        pdf.set_font("Helvetica", "I", 9)
        pdf.multi_cell(0, 5, f"Raw stats: {json.dumps(stats, default=str)}")

        out_path = os.path.join(config.REPORTS_DIR, "gta_v_report.pdf")
        pdf.output(out_path)
        return out_path

    def send_email(self, report_path: str, insights: str, recipient: str = None) -> bool:
        recipient = recipient or config.RECIPIENT_EMAIL
        if not config.EMAIL_APP_PASSWORD:
            raise RuntimeError("EMAIL_APP_PASSWORD is not set.")

        msg = MIMEMultipart()
        msg["From"] = config.EMAIL_ADDRESS
        msg["To"] = recipient
        msg["Subject"] = "Your Multi-Agent GTA V Sales Report"
        msg.attach(MIMEText(f"Hi,\n\nHere is the latest automated report.\n\n{insights}\n", "plain"))

        with open(report_path, "rb") as f:
            part = MIMEApplication(f.read(), _subtype="pdf")
            part.add_header("Content-Disposition", "attachment", filename=os.path.basename(report_path))
            msg.attach(part)

        with smtplib.SMTP(config.SMTP_SERVER, config.SMTP_PORT) as server:
            server.starttls()
            server.login(config.EMAIL_ADDRESS, config.EMAIL_APP_PASSWORD)
            server.sendmail(config.EMAIL_ADDRESS, recipient, msg.as_string())
        return True

    def update_dashboard(self, stats: dict, insights: str, fx_rate) -> str:
        payload = {
            "updated_at": datetime.datetime.now().isoformat(timespec="seconds"),
            "stats": stats,
            "insights": insights,
            "usd_to_egp": fx_rate,
        }
        with open(config.DASHBOARD_STATE_PATH, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, default=str)
        return config.DASHBOARD_STATE_PATH

Overwriting agents/action_agent.py


In [28]:
%%writefile orchestrator.py
import time
import threading

from agents.retrieval_agent import RetrievalAgent
from agents.analysis_agent import AnalysisAgent
from agents.action_agent import ActionAgent


class Orchestrator:
    def __init__(self):
        self.retrieval = RetrievalAgent()
        self.analysis = AnalysisAgent()
        self.action = ActionAgent()

    def run(self, csv_path: str, send_email: bool = True, recipient: str = None) -> dict:
        log = {"steps": {}, "success": True}

        t0 = time.time()
        try:
            data = self.retrieval.retrieve_all(csv_path)
            log["steps"]["retrieval"] = {"ok": True, "seconds": round(time.time() - t0, 2)}
        except Exception as e:
            log["steps"]["retrieval"] = {"ok": False, "error": str(e)}
            log["success"] = False
            return log

        t0 = time.time()
        try:
            stats = self.analysis.compute_stats(data["sales_df"], data["columns"])
            insights = self.analysis.generate_insights(stats, data["usd_to_egp"])
            log["steps"]["analysis"] = {"ok": True, "seconds": round(time.time() - t0, 2)}
        except Exception as e:
            log["steps"]["analysis"] = {"ok": False, "error": str(e)}
            log["success"] = False
            return log

        t0 = time.time()
        results = {}

        def _report():
            results["report_path"] = self.action.generate_report(stats, insights)

        def _dashboard():
            results["dashboard_path"] = self.action.update_dashboard(stats, insights, data["usd_to_egp"])

        threads = [threading.Thread(target=_report), threading.Thread(target=_dashboard)]
        for t in threads:
            t.start()
        for t in threads:
            t.join()

        email_ok = False
        if send_email and "report_path" in results:
            try:
                email_ok = self.action.send_email(results["report_path"], insights, recipient)
            except Exception as e:
                results["email_error"] = str(e)

        log["steps"]["actions"] = {
            "ok": "report_path" in results and "dashboard_path" in results,
            "seconds": round(time.time() - t0, 2),
            "email_sent": email_ok,
        }
        log["stats"] = stats
        log["insights"] = insights
        log["report_path"] = results.get("report_path")
        log["dashboard_path"] = results.get("dashboard_path")
        return log

Overwriting orchestrator.py


In [29]:
%%writefile evaluate.py
from orchestrator import Orchestrator
import config


def run_evaluation(csv_path: str = config.DEFAULT_CSV_PATH, runs: int = 3):
    orch = Orchestrator()
    results = [orch.run(csv_path, send_email=False) for _ in range(runs)]

    successes = sum(1 for r in results if r["success"])
    avg_times = {}
    for step in ["retrieval", "analysis", "actions"]:
        times = [r["steps"][step]["seconds"] for r in results if step in r["steps"] and r["steps"][step].get("ok")]
        if times:
            avg_times[step] = round(sum(times) / len(times), 2)

    summary = {"runs": runs, "success_rate": f"{successes}/{runs}", "avg_seconds_per_step": avg_times}
    return summary, results


if __name__ == "__main__":
    summary, _ = run_evaluation()
    print(summary)

Overwriting evaluate.py


In [30]:
%%writefile app.py
import json
import os

import pandas as pd
import plotly.express as px
import streamlit as st

import config
from orchestrator import Orchestrator
from evaluate import run_evaluation

st.set_page_config(page_title="GTA V Multi-Agent Analytics", page_icon="🎮", layout="wide")

st.markdown("""
<style>
.main {background-color: #0e1117;}
h1, h2, h3 {font-family: 'Segoe UI', sans-serif;}
.stButton>button {border-radius: 8px; font-weight: 600;}
</style>
""", unsafe_allow_html=True)

st.title("🎮 GTA V Worldwide Sales — Multi-Agent AI System")
st.caption("Retrieval Agent → Analysis Agent (LLM) → Action Agent (email / report / dashboard)")

with st.sidebar:
    st.header("Pipeline Controls")
    # Removed file uploader as per user request
    csv_path = config.DEFAULT_CSV_PATH # Use the default CSV path
    st.write(f"Loading data from: **{csv_path}**")

    recipient = st.text_input("Send report to", value=config.RECIPIENT_EMAIL)
    send_email = st.checkbox("Send email on run", value=True)
    run_btn = st.button("▶ Run Multi-Agent Pipeline", use_container_width=True)
    eval_btn = st.button("🧪 Run Evaluation (3x)", use_container_width=True)

tab_overview, tab_insights, tab_actions, tab_eval = st.tabs(
    ["📊 Data Overview", "🧠 AI Insights", "⚙️ Automated Actions", "✅ Evaluation"]
)

if run_btn:
    if not os.path.exists(csv_path):
        st.error(f"CSV not found at {csv_path}. Please ensure the file is present.")
    else:
        status = st.status("Running multi-agent pipeline...", expanded=True)
        status.write("🔎 Retrieval Agent: loading CSV + live FX rate...")
        orch = Orchestrator()
        result = orch.run(csv_path, send_email=send_email, recipient=recipient)
        if result["success"]:
            status.write("🧠 Analysis Agent: stats computed, Claude generated insights.")
            tail = ", email sent." if result["steps"]["actions"]["email_sent"] else "."
            status.write("⚙️ Action Agent: report + dashboard updated" + tail)
            status.update(label="Pipeline complete ✅", state="complete")
            st.session_state["last_result"] = result
        else:
            status.update(label="Pipeline failed ❌", state="error")
            st.json(result)

if eval_btn:
    with st.spinner("Running pipeline 3 times for evaluation..."):
        summary, runs = run_evaluation(csv_path)
    st.session_state["eval_summary"] = summary

result = st.session_state.get("last_result")

with tab_overview:
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        st.dataframe(df.head(20), use_container_width=True)
        numeric_cols = df.select_dtypes("number").columns.tolist()
        if numeric_cols:
            col = st.selectbox("Chart a numeric column", numeric_cols)
            st.plotly_chart(px.histogram(df, x=col, nbins=30, title=f"Distribution of {col}"), use_container_width=True)
    else:
        st.info("Data not loaded. Please ensure the default CSV file exists.")

with tab_insights:
    if result:
        st.subheader("Executive Summary (generated by Claude)")
        st.write(result["insights"])
        st.subheader("Underlying stats")
        st.json(result["stats"])
    else:
        st.info("Run the pipeline to generate AI insights.")

with tab_actions:
    if result:
        c1, c2, c3 = st.columns(3)
        c1.metric("Report", "✅ Generated" if result.get("report_path") else "—")
        c2.metric("Dashboard", "✅ Updated" if result.get("dashboard_path") else "—")
        c3.metric("Email", "✅ Sent" if result["steps"]["actions"]["email_sent"] else "Not sent")
        if result.get("report_path") and os.path.exists(result["report_path"]):
            with open(result["report_path"], "rb") as f:
                st.download_button("⬇ Download PDF report", f, file_name="gta_v_report.pdf")
        if os.path.exists(config.DASHBOARD_STATE_PATH):
            with open(config.DASHBOARD_STATE_PATH) as f:
                st.json(json.load(f))
    else:
        st.info("Run the pipeline to trigger automated actions.")

with tab_eval:
    summary = st.session_state.get("eval_summary")
    if summary:
        st.subheader("Reliability & Efficiency")
        st.write(f"Success rate: **{summary['success_rate']}**")
        st.json(summary["avg_seconds_per_step"])
    else:
        st.info("Click 'Run Evaluation' in the sidebar to test reliability & speed across multiple runs.")

Overwriting app.py


In [31]:
!pkill -f streamlit
import subprocess, time

log_file = open("streamlit_log.txt", "w")
proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=log_file, stderr=subprocess.STDOUT
)
time.sleep(15)
!cat streamlit_log.txt



2026-09-20 22:54:10.580 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.132.144:8501



In [32]:
!pkill -f streamlit
!pkill -f lt

In [33]:
import subprocess

try:
    # Run the evaluation script and capture its output
    result = subprocess.run(['python', 'evaluate.py'], capture_output=True, text=True, check=True)
    print("Evaluation Summary:")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print(f"Error during evaluation: {e.stderr}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Evaluation Summary:
{'runs': 3, 'success_rate': '0/3', 'avg_seconds_per_step': {'retrieval': 0.23}}



### Fixing Streamlit serving issues with `ngrok`

The previous `TypeError` messages indicate that `localtunnel` might be intermittently failing to fetch necessary JavaScript modules for the Streamlit application. We will switch to `ngrok` for more stable tunneling.

First, we need to install `pyngrok`.

In [34]:
!pip install -q pyngrok

Next, we'll set up `ngrok` with your authentication token. If you haven't already, please obtain an `ngrok` authentication token from your `ngrok` dashboard and add it to Colab's secrets manager (the '🔑' icon in the left panel) under the name `NGROK_AUTH_TOKEN`.

In [35]:
from pyngrok import ngrok, conf
import os

# Terminate any existing ngrok tunnels
ngrok.kill()

# Get ngrok auth token (replace with your actual token)
NGROK_AUTH_TOKEN = "3JI3Jz97kMvfy28ZaHKwMMDcGlv_2UeNn1pAEHhcFpY9BFVs7" # PASTE YOUR NGROK AUTH TOKEN HERE

# Configure ngrok with the auth token
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN_HERE":
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print("ngrok auth token configured.")
else:
    print("Warning: NGROK_AUTH_TOKEN not provided. ngrok might not work without authentication.")
    print("Please replace 'YOUR_NGROK_AUTH_TOKEN_HERE' with your actual ngrok auth token.")

ngrok auth token configured.


Now we will start the Streamlit app and expose it via `ngrok`. This replaces the previous cell that used `localtunnel`.

In [36]:
import subprocess
import time

# Start Streamlit in the background
streamlit_process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# Give Streamlit a moment to start up
time.sleep(5)

# Open an ngrok tunnel to the Streamlit app
try:
    public_url = ngrok.connect(8501)
    print(f"Streamlit App URL: {public_url}")
except Exception as e:
    print(f"Failed to start ngrok tunnel: {e}")
    print("Please ensure your NGROK_AUTH_TOKEN is correctly set in Colab secrets.")

Streamlit App URL: NgrokTunnel: "https://catcall-cultural-scribble.ngrok-free.dev" -> "http://localhost:8501"
